## Exercises on Generalizing with Neural Networks

These paper-and-pencil exercises reinforce Chapter 07. Exercises 7.1–7.7 cover the first part (NFQ): why tabular methods fail in large/continuous spaces, value functions as parameterised approximators $Q(s,a;\theta)$, the mean-squared TD loss, the **semi-gradient** update (treating the bootstrapped target as a constant), mini-batch loss, the handling of terminal and truncated transitions, the *fitted* (fixed-target) iteration, and the non-stationary-target problem. Exercises 7.8–7.12 cover the second part (DQN and its refinements): the two targets of DQN vs. Double DQN, why a replay buffer decorrelates data, prioritised experience replay (priorities, sampling probabilities, importance-sampling weights), the dueling aggregation and its identifiability constraint, and Polyak (soft) target updates. Notation: $\theta$ are the approximator parameters ($\theta^-$ those of the target network), $\phi(s,a)$ the feature vector of a linear approximator, $y$ the bootstrapped target, $\delta = y - Q$ the TD error, $\alpha$ the learning rate, $\gamma$ the discount, $d$ the terminal flag.


### Exercise 7.1 — The cost of discretising a continuous state space

The Cart-Pole state has $d=4$ continuous variables and $|\mathcal{A}|=2$ actions. Suppose we discretise each state variable into $b$ bins and store a tabular $Q$.

1. How many state cells and how many $Q$-table entries result, as a function of $b$?
2. Evaluate for $b=10$ and $b=20$.
3. Two of the four variables (the velocities) are unbounded — comment on the additional difficulty.

**Step 1 — Count.** With $b$ bins per dimension and $d$ dimensions, the number of discrete states is $b^{d}$; the $Q$-table has one entry per (state, action):

$\displaystyle \#\text{cells}=b^{d}, \qquad \#\text{entries}=|\mathcal{A}|\,b^{d}.$

**Step 2 — Evaluate** ($d=4,\ |\mathcal{A}|=2$):

| $b$ | cells $b^4$ | entries $2b^4$ |
|---|---|---|
| $10$ | $10000$ | $20000$ |
| $20$ | $160000$ | $320000$ |

**Step 3 — Unbounded variables.** The cart and pole *velocities* range over $(-\infty,\infty)$, so any finite binning must first **clip** them to an arbitrary range; states outside the range collapse into edge bins, losing information. Finer binning (larger $b$) reduces this error but makes $b^d$ explode — the **curse of dimensionality**. Most cells are also visited rarely or never, so the table cannot be filled reliably.

**Key concept**

Tabular representation scales as $b^{d}$: exponential in the number of state variables, and impossible for continuous states without discretisation. Note that for $d=4$ memory is not the real obstacle (a few hundred thousand entries are nothing); the obstacles are the **time needed to visit every cell often enough** and the **absence of generalisation**: two adjacent cells share no information, so what is learned in one tells nothing about the other. This motivates **function approximation**, which *generalises* across nearby states instead of storing each one.


### Exercise 7.2 — A semi-gradient update with a linear approximator

Use a linear action-value approximator $Q(s,a;\theta)=\theta^\top\phi(s,a)$ with weights $\theta=(0.2,-0.1,0.5)$ and feature vector $\phi(s,a)=(1,0,2)$ for the current pair. A transition gives reward $r=1$, $\gamma=0.9$, and the greedy next-state value is $\max_{a'}Q(s',a';\theta)=2.0$. Learning rate $\alpha=0.1$.

1. Compute the current estimate $Q(s,a;\theta)$.
2. Form the (bootstrapped) TD target and the TD error $\delta$.
3. Perform one **semi-gradient** update $\theta\leftarrow\theta+\alpha\,\delta\,\phi(s,a)$ and recompute $Q(s,a)$.

**Step 1 — Current estimate.** $\ Q(s,a;\theta)=\theta^\top\phi = 0.2(1)+(-0.1)(0)+0.5(2) = 1.2.$

**Step 2 — TD target and error.** The target is treated as a fixed number (it is *detached* from $\theta$):

$\displaystyle \text{target} = r + \gamma\max_{a'}Q(s',a';\theta) = 1 + 0.9(2.0) = 2.8, \qquad \delta = \text{target}-Q = 2.8-1.2 = 1.6.$

**Step 3 — Semi-gradient step.** For a linear model $\nabla_\theta Q(s,a;\theta)=\phi(s,a)$, so

$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,\phi = (0.2,-0.1,0.5) + 0.1(1.6)(1,0,2) = (0.36, -0.1, 0.82).$

New estimate: $\ Q(s,a;\theta)=\theta^\top\phi = 2$, which has moved from $1.2$ toward the target $2.8$ (by exactly $\alpha\delta\|\phi\|^2 = 0.1(1.6)(5)=0.8$).

**Key concept**

Function approximation replaces a table cell by a *weighted feature sum*; the update nudges the shared weights, so **every state that shares features with $(s,a)$ is affected** — this is generalisation. The gradient of a linear model is just its feature vector.

### Exercise 7.3 — Why "semi-gradient"? (target treated as constant)

The NFQ/TD loss for one transition is $\mathcal{L}(\theta)=\tfrac12\big(y-Q(s,a;\theta)\big)^2$, where the target $y=r+\gamma\max_{a'}Q(s',a';\theta)$ **also depends on $\theta$** (it is produced by the same network). Derive the gradient (i) treating $y$ as a constant (the semi-gradient used in practice), and (ii) differentiating $y$ as well (the full gradient), and explain why the semi-gradient is used.

**Step 1 — Semi-gradient (target detached).** Treating $y$ as a constant label:

$\displaystyle \nabla_\theta \mathcal{L} = -\big(y - Q(s,a;\theta)\big)\,\nabla_\theta Q(s,a;\theta) = -\,\delta\,\nabla_\theta Q(s,a;\theta).$

This gives the update $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta Q(s,a;\theta)$ used in Exercise 7.2.

**Step 2 — Full gradient.** If we also differentiate $y=r+\gamma\max_{a'}Q(s',a';\theta)$ (assume the max is attained at $a'^*$):

$\displaystyle \nabla_\theta \mathcal{L} = -\big(y-Q(s,a;\theta)\big)\Big[\nabla_\theta Q(s,a;\theta) - \gamma\,\nabla_\theta Q(s',a'^*;\theta)\Big].$

The extra term $+\gamma\,\delta\,\nabla_\theta Q(s',a'^*;\theta)$ is what the semi-gradient **drops**.

**Step 3 — Why drop it.** The full gradient defines a legitimate algorithm, the *residual-gradient* method (Baird, 1995), which minimises the mean squared Bellman error. It is not used in practice for two reasons. First, when the transitions are stochastic the sampled full gradient is **biased**: the term $\nabla_\theta Q(s',a'^*;\theta)$ and the error $\delta$ are computed from the *same* sampled $s'$, so their product does not estimate the product of the expectations; an unbiased estimate would need two independent samples of $s'$ from the same $(s,a)$ (the *double-sampling problem*), which a real environment does not provide. Second, even in deterministic environments the method is much slower, because the second term pulls the value of $s'$ *toward* $Q(s,a)$ while the first pulls $Q(s,a)$ toward $s'$, and the two effects partially cancel. The semi-gradient avoids both issues by treating $y$ as a supervised label: each update is a step of an ordinary regression toward a fixed target, which is exactly the *fitted* viewpoint of NFQ. The price is that the semi-gradient is not the gradient of any fixed objective, so the convergence guarantees of gradient descent do not apply to it.


**Key concept**

In supervised learning the target is a fixed label; in bootstrapped RL the target is produced by the very parameters being trained. Pretending it is constant (the **semi-gradient**) is what makes the update well-behaved — and is *the* recurring subtlety of value-based deep RL.

### Exercise 7.4 — Mean-squared loss over a mini-batch

A mini-batch of three transitions has current predictions $Q(s_i,a_i;\theta)=(1.2,\,0.5,\,2.0)$ and (fixed, detached) targets $y_i=(2.8,\,0.5,\,1.0)$. Compute the mean-squared TD loss and state the sign of the update each sample induces.

**Step 1 — Per-sample TD errors** $\delta_i=y_i-Q_i$:

$\displaystyle \delta_1 = 2.8-1.2 = 1.6,\qquad \delta_2 = 0.5-0.5 = 0,\qquad \delta_3 = 1.0-2.0 = -1.0.$

**Step 2 — Mean-squared loss.**

$\displaystyle \mathcal{L} = \frac{1}{3}\sum_{i=1}^{3}\delta_i^2 = \frac{1.6^2 + 0^2 + (-1.0)^2}{3} = \frac{2.56 + 0 + 1.0}{3} = 1.1867.$

**Step 3 — Update directions.** The semi-gradient step moves each prediction in the direction of its target (by exactly $\alpha\delta_i$ only in the tabular case, where $\nabla_\theta Q$ is a one-hot vector): sample 1 ($\delta_1>0$) is pushed **up**, sample 3 ($\delta_3<0$) is pushed **down**, and sample 2 ($\delta_2=0$) contributes nothing. Because the weights are shared, the net gradient is the average of the per-sample gradients.

**Key concept**

Mini-batch training averages the gradient over several transitions, reducing variance and smoothing the update — but the samples must be reasonably **independent** for this to behave like supervised learning, which sets up the correlation problem addressed next.

### Exercise 7.5 — Chasing a moving target (non-stationarity)

Consider a scalar linear approximator $Q(s;\theta)=\theta\,x(s)$ with a single feature. Two adjacent states share the *same* feature value $x(s)=x(s')=1$ (they look identical to the approximator). A transition $s\to s'$ gives $r=0$, $\gamma=0.9$; start with $\theta=1$ and $\alpha=0.5$.

1. Compute the TD target for this transition, and perform one semi-gradient update of $\theta$.
2. Recompute the target for the **same** transition after the update, and explain what happened.

**Step 1 — Target and update.** Current values: $Q(s)=\theta x(s)=1$, $Q(s')=\theta x(s')=1$.

$\displaystyle \text{target} = r + \gamma Q(s') = 0 + 0.9(1) = 0.9, \qquad \delta = 0.9 - 1 = -0.1.$
$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,x(s) = 1 + 0.5(-0.1)(1) = 0.95.$

**Step 2 — Recompute the target after the update.** Because $s$ and $s'$ share the feature, updating $\theta$ to fit $s$ also changed $Q(s')$:

$\displaystyle \text{target}' = r + \gamma Q(s';\theta_{\text{new}}) = 0.9\,(0.95)(1) = 0.855.$

The target moved from $0.9$ to $0.855$ — we stepped *toward* a target that then *shifted away*.

**Step 3 — Interpretation.** With a shared/generalising representation, an update aimed at $Q(s)$ perturbs $Q(s')$, so the bootstrapped target is **non-stationary**: it changes every time we train. This "chasing your own tail" is the mechanism behind the instability of naive value-based deep RL.

**Step 4 — Does this example diverge?** Iterating the update on the same transition gives

$\displaystyle \theta_{k+1} = \theta_k + \alpha\,(\gamma\theta_k - \theta_k) = \big(1 - \alpha(1-\gamma)\big)\,\theta_k = 0.95\,\theta_k \ \longrightarrow\ 0,$

so here the chase *converges*, and to the right value: with $r=0$ the true value of both states is $0$. Non-stationarity of the target is not by itself divergence. Divergence requires the full **deadly triad** (function approximation, bootstrapping, off-policy data); in this example the data is on-policy and the contraction factor $\gamma<1$ still wins. Baird's counterexample (Sutton & Barto, §11.2) shows that with off-policy updates the same kind of linear approximator can diverge to infinity.


**Key concept**

Function approximation couples states through shared parameters, making the learning target move as the weights move. In favourable cases the moving target still converges; in the off-policy setting of Q-learning it can diverge. Stabilising the target — e.g. with a **frozen target network** updated only occasionally — is precisely the fix introduced by DQN in the next chapter.


### Exercise 7.6 — Terminal and truncated transitions

In the Cart-pole every step gives $r=1$ and $\gamma=0.99$. For a transition $(s,a,r,s')$ the network currently estimates $\max_{a'}Q(s',a';\theta)=50$. Compute the target $y = r + \gamma\,(1-d)\max_{a'}Q(s',a';\theta)$ in three cases:

1. the pole falls at $s'$ (the episode **terminates**);
2. the episode is **truncated** at $s'$ because step 500 has been reached;
3. as in case 2, but the implementation wrongly sets $d=1$ whenever the episode ends (`done = terminated or truncated`).

Then explain what the network learns in case 3, keeping in mind that the time step is *not* part of the Cart-pole state, and compute the value of a state in which the pole is balanced forever.


**Step 1 — Terminated.** $s'$ is terminal, $d=1$: no future reward is possible.

$\displaystyle y = 1 + 0.99\,(1-1)\,50 = 1.$


**Step 2 — Truncated.** $s'$ is *not* terminal, $d=0$: the agent could have continued, so we bootstrap.

$\displaystyle y = 1 + 0.99\,(1-0)\,50 = 50.5.$


**Step 3 — Truncation treated as termination.** With $d=1$ the target is again $y=1$: the same state $s'$ is labelled as worth $50.5$ when reached at step 200 and as worth $1$ when reached at step 500. Since the time step is not observed, the two situations are **the same input** for the network, and the regression is asked to fit two contradictory labels. The effect is not confined to the last state: a well-balanced pole at step 500 looks exactly like a well-balanced pole at any other step, so the artificially low value spreads by generalisation to *all* good states, holding down the values the agent is trying to learn and slowing learning considerably.


**Step 4 — Value of a perfectly balanced state.** If the pole never falls the agent receives $r=1$ forever, and the discounted return is

$\displaystyle q_*(s,a) = \sum_{k=0}^{\infty} \gamma^{k} = \frac{1}{1-\gamma} = \frac{1}{0.01} = 100.$

This is the value the network should converge to for good states, and it is why the estimate $50$ in the exercise is plausible for a partially trained agent. The undiscounted return of an episode, instead, is capped at $500$ by the truncation: the *score* we plot and the *values* the network learns live on different scales.


**Key concept**

A **terminal** state has no future value ($d=1$); a **truncated** episode simply stops observing a state that does have future value ($d=0$). Confusing the two injects a label that depends on an unobservable quantity (time) and, through generalisation, corrupts the values of all similar states. In code, use `terminated` for the flag, never `terminated or truncated`.


### Exercise 7.7 — Fixed targets vs. recomputed targets

Take again the scalar approximator of Exercise 7.5: $Q(s;\theta)=\theta\,x(s)$ with $x(s)=x(s')=1$, the single transition $s\to s'$ with $r=0$, $\gamma=0.9$, initial $\theta_0=1$, $\alpha=0.5$. The "batch" contains this one transition and is used for **two epochs** of optimisation.

1. **Fitted (NFQ) variant:** compute the target once from $\theta_0$, then perform two semi-gradient steps toward that *fixed* target.
2. **Moving-target variant:** recompute the target from the current $\theta$ before each of the two steps.
3. Compare the two results, and describe what happens to the loss $\mathcal{L}(\theta)=\tfrac12(y-Q)^2$ in the two variants as the number of epochs grows.


**Step 1 — Fixed target.** Computed once: $\ y = r + \gamma\,\theta_0 = 0.9$, and held constant.

$\displaystyle \text{epoch 1:}\quad \delta = 0.9 - 1 = -0.1, \qquad \theta_1 = 1 + 0.5(-0.1) = 0.95;$

$\displaystyle \text{epoch 2:}\quad \delta = 0.9 - 0.95 = -0.05, \qquad \theta_2 = 0.95 + 0.5(-0.05) = 0.925.$

The parameter moves toward $0.9$ and, with more epochs, converges exactly to it: $\theta_k - 0.9 = 0.5^{k}\,(\theta_0-0.9)$. The loss decreases monotonically to $0$, as in any well-posed regression.


**Step 2 — Recomputed target.**

$\displaystyle \text{epoch 1:}\quad y = 0.9\,\theta_0 = 0.9, \quad \delta = -0.1, \qquad \theta_1 = 0.95;$

$\displaystyle \text{epoch 2:}\quad y = 0.9\,\theta_1 = 0.855, \quad \delta = 0.855 - 0.95 = -0.095, \qquad \theta_2 = 0.95 + 0.5(-0.095) = 0.9025.$

Each epoch is now an iteration of Exercise 7.5: $\theta_k = 0.95^k\,\theta_0 \to 0$.


**Step 3 — Comparison.** After two epochs the two variants give $\theta_2 = 0.925$ and $\theta_2 = 0.9025$. In this deterministic toy problem both are moving in the right direction (the true value is $0$), and the moving-target variant is even faster, because it performs a Bellman backup at every epoch instead of every batch. The difference lies in *what the optimisation is doing*. With fixed targets each fit is a **well-posed supervised problem**: the loss has a fixed minimiser ($\theta = y$), it decreases monotonically, and reaching a small loss means the network has *fitted the targets*; the Bellman backup happens once per batch, when the targets are recomputed, and it is a contraction ($y_{\text{new}} = \gamma\,\theta$) exactly as in value iteration. With recomputed targets the loss has **no fixed minimiser**: the quantity being minimised changes at every step, so a small loss means only that $Q$ is close to the *current* target, not that the fit has converged, and with a neural network (many parameters, correlated samples, large steps) the target can move faster than the network follows it. This is why NFQ, like fitted Q iteration in general, freezes the targets for the duration of each fit.


**Key concept**

"Fitted" means: compute the targets once with the current network, then solve an ordinary regression toward these **fixed** labels; only then recompute the targets. Each fit is well-posed, and the Bellman backup is confined to the moments in which the targets are refreshed. DQN's target network takes the idea one step further by freezing the targets across *many* batches.


### Exercise 7.8 — DQN vs. Double DQN targets

For a sampled transition with reward $r=1$ and $\gamma=0.9$, the next state $s'$ has action-values

$\displaystyle Q(s',\cdot;\theta)=(1.0,\ 3.0)\ \text{(online)}, \qquad Q(s',\cdot;\theta^-)=(2.5,\ 2.0)\ \text{(target)}.$

Compute the **DQN** target and the **Double DQN** target, and explain why DDQN reduces overestimation.

**Step 1 — DQN target.** DQN takes the **max over the target network** (selection *and* evaluation by $\theta^-$):

$\displaystyle y_{\text{DQN}} = r + \gamma\max_{a'}Q(s',a';\theta^-) = 1 + 0.9\max(2.5,2.0) = 1 + 0.9(2.5) = 3.25.$

**Step 2 — Double DQN target.** DDQN **selects** the action with the online network and **evaluates** it with the target network:

$\displaystyle a^* = \arg\max_{a'}Q(s',a';\theta) = a_2 \ (\text{online value }3.0),$
$\displaystyle y_{\text{DDQN}} = r + \gamma\,Q(s',a^*;\theta^-) = 1 + 0.9\,Q(s',a_2;\theta^-) = 1 + 0.9(2.0) = 2.8.$

**Step 3 — Why lower.** The online argmax is $a_2$, but the *target* network rates $a_2$ at only $2.0$ (not the $2.5$ that DQN's own max would grab from $a_1$). By forcing the evaluation to come from an independent network, DDQN avoids automatically pairing "the action that looks best" with "the value that looks biggest", so $y_{\text{DDQN}}=2.8 < 3.25 = y_{\text{DQN}}$.

**Key concept**

DQN's single max both *selects* and *evaluates* with the same (noisy) estimates, producing the maximisation bias of Chapter 05 — now in a deep setting. DDQN decouples the two using the target network the algorithm already maintains, at no extra cost.

### Exercise 7.9 — Replay buffer and decorrelation

A replay buffer holds $N=100$ transitions; each optimisation step samples a mini-batch of $m=10$ **without replacement**, uniformly at random.

1. What is the probability that a specific transition is included in a given mini-batch?
2. Two *temporally consecutive* transitions (collected at steps $t$ and $t{+}1$) are strongly correlated. What is the probability that **both** land in the same mini-batch?
3. Interpret why this addresses the IID-violation problem of online updates.

**Step 1 — Inclusion probability.** With uniform sampling without replacement, each transition is equally likely to be among the $m$ chosen out of $N$:

$\displaystyle \Pr(\text{a specific transition in the batch}) = \frac{m}{N} = \frac{10}{100} = 0.1.$

**Step 2 — Both correlated samples together.** The first of the pair is included with prob. $m/N$; given that, the second occupies one of the remaining $m-1$ of the remaining $N-1$ slots:

$\displaystyle \Pr(\text{both}) = \frac{m}{N}\cdot\frac{m-1}{N-1} = \frac{10}{100}\cdot\frac{9}{99} = 0.0091\ (\approx 0.9\%).$

**Step 3 — Interpretation.** Online updates use consecutive, highly correlated transitions every step, badly violating the IID assumption of stochastic-gradient training. Sampling from a large buffer makes it *unlikely* that adjacent, correlated transitions are learned together (here under $1\%$), so each mini-batch mixes experiences from many different times and policies — approximately IID.

**Key concept**

Experience replay converts a correlated online stream into a near-IID dataset by **breaking temporal adjacency**, which is what lets standard gradient-based optimisation remain stable. It also lets each transition be reused many times, improving sample efficiency.

### Exercise 7.10 — Prioritised Experience Replay

Four transitions in the buffer have TD errors $\delta=(2.0,\,1.0,\,0.5,\,0.0)$. Prioritised replay uses priority $p_i=|\delta_i|+\epsilon$ with $\epsilon=0.5$, sampling exponent $\alpha=1$, and importance-sampling exponent $\beta=0.5$ (buffer size $N=4$).

1. Compute the priorities and the sampling probabilities $P(i)=p_i^{\alpha}/\sum_k p_k^{\alpha}$.
2. Compute the importance-sampling weights $w_i=\big(\tfrac1N\tfrac1{P(i)}\big)^{\beta}$, normalised so the largest is $1$.
3. Explain the role of $\epsilon$ and of the IS weights.

**Step 1 — Priorities and probabilities.** $p_i=|\delta_i|+0.5 = (2.5,\,1.5,\,1,\,0.5)$, sum $=5.5$. With $\alpha=1$:

$\displaystyle P = \Big(0.4545,\ 0.2727,\ 0.1818,\ 0.0909\Big).$

**Step 2 — Importance-sampling weights.** $w_i=\big(N\,P(i)\big)^{-\beta}$ with $N=4,\ \beta=0.5$:

$\displaystyle w = (0.7416,\,0.9574,\,1.1726,\,1.6583)\ \xrightarrow{\div\max} \ \tilde w = (0.4472,\,0.5774,\,0.7071,\,1).$

**Step 3 — Interpretation.** The constant $\epsilon$ keeps the zero-TD-error transition (transition 4) at a **non-zero** priority ($P=0.0909>0$), so it can still be replayed — without it, a transition that once had $\delta=0$ would never be sampled again. The IS weights **counteract the sampling bias**: the most-sampled (highest-priority) transition 1 gets the *smallest* weight $0.4472$, while the least-sampled transition 4 gets weight $1$, so that frequently-replayed transitions do not dominate the gradient.

**Key concept**

PER replays *surprising* (high-$|\delta|$) transitions more often to learn faster, but this skews the data distribution; importance-sampling weights re-balance the updates so the expected gradient stays (approximately) correct. $\alpha$ tunes how greedy the prioritisation is; $\beta$ tunes how fully the bias is corrected.

### Exercise 7.11 — Dueling architecture and identifiability

A dueling network outputs a state-value $V(s)=5$ and raw advantages $A(s,\cdot)=(1,\,-1,\,2)$ for three actions. It combines them with the **mean-subtracted** aggregation

$\displaystyle Q(s,a) = V(s) + \Big(A(s,a) - \tfrac{1}{|\mathcal{A}|}\sum_{a'}A(s,a')\Big).$

1. Compute $Q(s,\cdot)$ and verify that $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$.
2. Show that the *naive* aggregation $Q=V+A$ is **unidentifiable**, and explain how the mean-subtraction fixes it.

**Step 1 — Aggregate.** Mean advantage $\bar A=\tfrac{1+(-1)+2}{3}=0.6667$. Then

$\displaystyle Q(s,\cdot) = 5 + \big[(1,-1,2)-0.6667\big] = (5.3333, 3.3333, 6.3333).$

Check: $\tfrac13\sum_a Q(s,a) = \tfrac{5.3333+3.3333+6.3333}{3} = 5 = V(s)$. ✓ The mean-subtraction forces the value head to equal the mean action-value.

**Step 2 — Unidentifiability of $Q=V+A$.** The same $Q(s,\cdot)=(5.3333, 3.3333, 6.3333)$ can be produced by infinitely many $(V,A)$ splits, e.g.

- $V=5,\ A=(0.3333,-1.6667,1.3333)$ (mean $0$), or
- $V=0,\ A=(5.3333,3.3333,6.3333)$ (mean $5$),

since only the *sum* $V+A$ enters $Q$. Given only $Q$, we cannot recover $V$ and $A$ separately — the decomposition is not unique.

**Step 3 — The fix.** Imposing $\sum_a A(s,a)=0$ (equivalently, subtracting the mean advantage) removes this freedom: it pins $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$ and $A(s,a)=Q(s,a)-V(s)$, a *unique* decomposition. This is why the aggregation subtracts $\bar A$ rather than using $V+A$ directly.

**Key concept**

The dueling network separately estimates "how good is this state" ($V$) and "how much better is each action" ($A$), sharing features so that learning about one action informs the state value. The mean-subtraction is not cosmetic — it is what makes the two heads *identifiable* and stably trainable.

### Exercise 7.12 — Polyak (soft) target updates

A target network is updated softly by $\theta^- \leftarrow \tau\,\theta + (1-\tau)\,\theta^-$ with mixing factor $\tau=0.1$. For a single scalar weight, suppose the online weight is fixed at $\theta=10$ and the target starts at $\theta^-_0=0$.

1. Compute $\theta^-$ after three soft updates.
2. Give the closed form $\theta^-_k$ and contrast soft updates with a hard periodic copy.

**Step 1 — Iterate** $\theta^-\leftarrow 0.1(10)+0.9\,\theta^-$:

$\displaystyle \theta^-_1 = 0.1(10)+0.9(0) = 1,\quad \theta^-_2 = 1+0.9(1) = 1.9,\quad \theta^-_3 = 1+0.9(1.9) = 2.71.$

**Step 2 — Closed form.** Unrolling with constant online weight $\theta=10$:

$\displaystyle \theta^-_k = \theta\big(1-(1-\tau)^k\big) = 10\big(1-0.9^{k}\big),$

giving $10(1-0.9)=1,\ 10(1-0.81)=1.9,\ 10(1-0.729)=2.71$ — matching Step 1. The target **exponentially approaches** the online weight, always lagging.

**Step 3 — Soft vs. hard.** A *hard* update copies $\theta$ into $\theta^-$ every $C$ steps: the target is perfectly stable between copies but jumps discontinuously. A *soft* (Polyak) update moves the target a little ($\tau$ small) every step: it lags smoothly, avoiding the large discrete jump while still changing slowly enough to stabilise the bootstrapped target.

**Key concept**

The target network trades learning speed for stability; Polyak averaging with small $\tau$ makes that trade **smooth and continuous** instead of abrupt. (Note the ordering: $\tau$ multiplies the *online* weights — a small $\tau$ means a *small* step toward the online network, i.e. a slow-moving target.)